In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("HousingData.csv")
print(df.head(3))
print(f"Total rows are {(df.shape)[0]} and total columns are {(df.shape[1])}")
print(f"Column names : {df.columns}")

# MEDV = Median value of houses in $1000s
# MEDV = 24.0 means the house price is $24,000.
# Thus our output variable is MEDV

FileNotFoundError: [Errno 2] No such file or directory: 'HousingData.csv'

In [ ]:
print(df.isnull().sum())

# Replacing null values with the median of their respective columns
df = df.fillna(df.median())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 12))
sns.heatmap(df.corr(), annot=True)
plt.show()


In [ ]:
# - `RM` usually has a positive correlation with `MEDV`, meaning houses with more rooms usually have higher prices.
# - `LSTAT` usually has a negative correlation with `MEDV`, meaning as lower-status population percentage increases, house price usually decreases.
# - The heatmap also helps detect multicollinearity, where two input features are highly related to each other.

In [ ]:
# Remaining data
# X contains independent variables/features used for prediction
# MEDV is removed because it is the value we want to predict
# Correlation of CHAS with MEDV is below around 0.25
# It means CHAS has a very weak relationship with house price.
X = df.drop(["MEDV", "CHAS"], axis=1)

# y contains the dependent variable/target
# The model will learn to predict this value
y = df["MEDV"]

# X → features
# y → target

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
# SPLit testnig and training dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=41
)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Before scaling
print('CRIM and TAX sample (first 3 rows) before scaling:')
print(X_train[['CRIM', 'TAX']].head(3))
print('\nCRIM and TAX means before scaling:')
print(X_train[['CRIM', 'TAX']].mean())

# ----------------------------------------------------------
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)
# ----------------------------------------------------------

# After scaling
df_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print('\nCRIM and TAX sample (first 3 rows) after scaling:')
print(df_train_scaled[['CRIM', 'TAX']].head(3))
print('\nCRIM and TAX means after scaling:')
print(df_train_scaled[['CRIM', 'TAX']].mean())

In [ ]:
# LInear regresison

model = LinearRegression()
model.fit(X_train, y_train)

# after scalar
# model.fit(X_train_scaled, y_train)

# Prediction step

# This is the actual prediction line
# The model uses X_test_scaled and predicts MEDV values for the test data

In [ ]:
y_pred = model.predict(X_test_scaled)
print("Predictions generated successfully.")

# Comparing orignal MEDV and predicted
comparison = pd.DataFrame({
    "Actual MEDV": y_test.values,
    "Predicted MEDV": y_pred
})
comparison

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Compare actual and predicted values
# y_test contains real MEDV values
# y_pred contains values predicted by the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# scope of error, smaller is better
print("RMSE:", rmse)

# correct percent, closer to 1 is better
print("R2 Score:", r2)

# Prev
# RMSE: 4.697783469023746
# R2 Score: 0.524688112403985

# After dropping CHAS
# RMSE: 4.5220660808105455
# R2 Score: 0.5595805458866143

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

print(coefficients.sort_values(by="Coefficient", ascending=False))

# This tells:
# Which features increase or decrease house price.
# It shows how much does each column affect out MEDV

# Example interpretation:
# Positive coefficient → increases price
# Negative coefficient → decreases price


In [ ]:
y_pred

In [ ]:
y_test

In [ ]:
# If predictions are perfect, points will lie close to the red diagonal line
# Points far from the red line show higher prediction error

plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red')
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
# Residual plot

# Residual means error = actual value - predicted value
# A good model should have residuals randomly scattered around 0
error = y_test - y_pred
residuals = y_test - y_pred

plt.scatter(y_pred, residuals)
plt.axhline(y=0, color='red')
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()


In [ ]:
# This plot shows how prediction errors are distributed
# If errors are mostly around 0, the model is performing reasonably well

sns.histplot(residuals, kde=True)
plt.title("Error Distribution")
plt.show()
